# P24 - Gym Rat Level Prediction Using KNN
**Complete Code | Data Included in Python | No CSV Upload Required**

This notebook predicts a gym member's **Experience Level** (1 = Beginner, 2 = Intermediate, 3 = Advanced) using the **K-Nearest Neighbours (KNN)** classification algorithm.

---
**Pipeline Steps:**
1. Dataset (hardcoded â€” no CSV needed)
2. Data Cleaning
3. Exploratory Data Analysis (EDA)
4. Feature Engineering
5. Train-Test Split
6. Feature Selection (SelectKBest)
7. Feature Scaling (StandardScaler)
8. Model Evaluation Function
9. Baseline KNN (k=5)
10. Hyperparameter Tuning (k, metric, scaling)
11. Final Model Selection by Macro F1
12. Predict Three Sample Inputs
13. Save Results to CSV

In [ ]:
# Uncomment if running on Google Colab
# !pip install scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix,
    ConfusionMatrixDisplay, classification_report,
    roc_auc_score
)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
print("Libraries imported successfully!")

## 1. Dataset

In [ ]:
data = {
    "Age": [22,25,28,35,40,21,24,30,32,27,45,38,29,23,26,31,34,42,20,37,
            28,33,41,22,36,25,30,39,44,27,21,24,35,40,29,32,26,38,43,23,
            31,34,28,45,22,37,25,33,41,30,20,26,35,39,24,32,43,27,36,22,
            29,40,31,25,38,34,21,44,28,37,23,30,42,26,35,32,24,39,27,41,
            22,33,36,28,45,25,31,38,20,34,29,43,24,37,30,26,40,32,21,35],
    "Gender": ["Male","Female","Male","Male","Female",
               "Female","Male","Female","Male","Female"] * 10,
    "Weight (kg)": [65,55,72,80,68,50,70,60,75,58,85,78,67,52,62,74,81,88,49,77,
                    71,83,69,54,79,61,73,86,90,57,51,64,82,76,68,72,59,84,91,53,
                    75,80,66,89,56,78,63,85,70,74,48,60,77,83,55,72,87,62,79,51,
                    68,90,73,58,82,76,49,88,65,80,54,71,86,61,75,69,57,84,63,92,
                    52,79,81,66,88,59,74,83,47,77,70,89,56,78,64,60,85,72,50,82],
    "Height (m)": [1.72,1.60,1.80,1.75,1.65,1.58,1.78,1.62,1.82,1.64] * 10,
    "Max_BPM": [170,165,180,175,168,160,178,172,185,166] * 10,
    "Avg_BPM": [130,120,145,138,125,115,140,128,150,122] * 10,
    "Resting_BPM": [65,70,60,68,72,75,62,67,58,73] * 10,
    "Session_Duration (hours)": [1.2,0.8,1.8,1.5,1.0,0.7,1.6,1.1,2.0,0.9] * 10,
    "Calories_Burned": [450,300,700,600,400,250,650,420,800,350] * 10,
    "Workout_Type": ["Cardio","Yoga","Strength","HIIT","Cardio",
                     "Yoga","Strength","Cardio","HIIT","Yoga"] * 10,
    "Fat_Percentage": [18,25,15,20,22,28,16,24,12,27] * 10,
    "Water_Intake (liters)": [3.0,2.0,3.5,3.2,2.5,1.8,3.4,2.6,4.0,2.1] * 10,
    "Workout_Frequency (days/week)": [5,2,6,5,3,1,6,4,7,2] * 10,
    "Experience_Level": [2,1,3,3,2,1,3,2,3,1] * 10
}

df = pd.DataFrame(data)
TARGET = "Experience_Level"

print("Dataset Loaded Successfully!")
print("Dataset Shape:", df.shape)
print("\nFirst Five Rows:")
display(df.head())
print("\nMissing Values:")
print(df.isnull().sum())
print("\nDuplicate Rows:", df.duplicated().sum())
print("\nTarget Class Counts:")
print(df[TARGET].value_counts())

## 2. Data Cleaning

In [ ]:
df = df.drop_duplicates().dropna(subset=[TARGET]).copy()

for col in df.columns:
    if col == TARGET:
        continue
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].fillna(df[col].median())
    else:
        mode = df[col].mode()
        df[col] = df[col].fillna(mode.iloc[0] if not mode.empty else "Unknown")

if df[TARGET].nunique() < 2:
    raise ValueError("At least two classes required.")

print("Cleaned Dataset Shape:", df.shape)
print("Total Missing Values:", df.isnull().sum().sum())

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Class distribution
plt.figure(figsize=(6, 4))
df[TARGET].value_counts().sort_index().plot(kind="bar", color="skyblue", edgecolor="black")
plt.title("Experience Level Class Counts")
plt.xlabel("Experience Level")
plt.ylabel("Number of Members")
plt.tight_layout()
plt.show()

# Histograms of all numeric features
numeric_df = df.select_dtypes(include=np.number)
numeric_df.hist(figsize=(14, 10), bins=15, edgecolor="black")
plt.suptitle("Histograms of Numeric Features")
plt.tight_layout()
plt.show()

# Boxplots vs Experience Level
for col in ["Session_Duration (hours)", "Workout_Frequency (days/week)"]:
    plt.figure(figsize=(7, 4))
    sns.boxplot(data=df, x=TARGET, y=col)
    plt.title(f"{col} by Experience Level")
    plt.tight_layout()
    plt.show()

# Correlation heatmap
corr_df = df.copy()
for col in corr_df.select_dtypes(include=["object", "category"]).columns:
    if col != TARGET:
        corr_df[col] = pd.factorize(corr_df[col].astype(str))[0]

plt.figure(figsize=(12, 8))
sns.heatmap(corr_df.corr(numeric_only=True), cmap="coolwarm", annot=True, fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

print("\nData Summary:")
display(df.describe(include="all").T)

## 4. Feature Engineering

In [ ]:
df_model = df.copy()
# New feature: water intake relative to body weight
df_model["Water_per_kg"] = (
    df_model["Water_Intake (liters)"] /
    df_model["Weight (kg)"].replace(0, np.nan)
).fillna(0)

X_raw = df_model.drop(columns=[TARGET])
y_raw = df_model[TARGET]

le_y = LabelEncoder()
y = le_y.fit_transform(y_raw.astype(str))

print("Target Label Mapping:", dict(enumerate(le_y.classes_)))

X = pd.get_dummies(X_raw, drop_first=True, dtype=float)
X = X.replace([np.inf, -np.inf], np.nan).fillna(0).astype(float)
print("Feature Matrix Shape:", X.shape)

## 5. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Training Rows:", len(X_train))
print("Testing Rows:",  len(X_test))

## 6. Feature Selection

In [ ]:
k_features = min(15, X_train.shape[1])
selector = SelectKBest(score_func=f_classif, k=k_features)
X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected  = selector.transform(X_test)

selected_features = X_train.columns[selector.get_support()].tolist()
print("Selected Features:")
print(selected_features)

## 7. Feature Scaling

In [ ]:
scaler = StandardScaler()
X_train_scaled   = scaler.fit_transform(X_train_selected)
X_test_scaled    = scaler.transform(X_test_selected)
X_train_unscaled = X_train_selected.copy()
X_test_unscaled  = X_test_selected.copy()
print("Scaling complete.")

## 8. Model Evaluation Function

In [ ]:
results = []

def evaluate(name, model, Xtr=None, Xte=None, details=False):
    if Xtr is None: Xtr = X_train_scaled
    if Xte is None: Xte = X_test_scaled

    model.fit(Xtr, y_train)
    pred   = model.predict(Xte)
    labels = np.arange(len(le_y.classes_))
    cm     = confusion_matrix(y_test, pred, labels=labels)

    specificities = []
    for i in range(len(labels)):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        specificities.append(tn / (tn + fp) if (tn + fp) > 0 else 0)

    auc = np.nan
    try:
        if len(np.unique(y_test)) == len(labels):
            probabilities = model.predict_proba(Xte)
            auc = roc_auc_score(
                y_test, probabilities,
                multi_class="ovr", average="macro", labels=labels
            )
    except (ValueError, IndexError):
        pass

    row = {
        "Setting":     name,
        "Accuracy":    accuracy_score(y_test, pred),
        "Precision":   precision_score(y_test, pred, average="macro", zero_division=0),
        "Recall":      recall_score(y_test, pred, average="macro", zero_division=0),
        "F1":          f1_score(y_test, pred, average="macro", zero_division=0),
        "Specificity": np.mean(specificities),
        "AUC":         auc
    }
    results.append(row)
    print("\n" + name)
    print(pd.Series(row).to_string())

    if details:
        ConfusionMatrixDisplay(
            confusion_matrix=cm, display_labels=le_y.classes_
        ).plot(cmap="Blues", values_format="d")
        plt.title(f"Confusion Matrix: {name}")
        plt.tight_layout()
        plt.show()
        print("\nClassification Report:")
        print(classification_report(
            y_test, pred, labels=labels,
            target_names=[str(c) for c in le_y.classes_],
            zero_division=0
        ))
    return row

## 9. Baseline KNN Model

In [ ]:
baseline_model = KNeighborsClassifier(n_neighbors=5)
evaluate("Baseline KNN (k=5, scaled)", baseline_model, details=True)

## 10. Hyperparameter Tuning

In [ ]:
k_values = [k for k in [1, 3, 5, 7, 9, 11, 13, 15] if k <= len(X_train)]

for k in k_values:
    evaluate(f"k={k}, euclidean, scaled",
             KNeighborsClassifier(n_neighbors=k, metric="euclidean"))
    evaluate(f"k={k}, manhattan, scaled",
             KNeighborsClassifier(n_neighbors=k, metric="manhattan"))

for k in [3, 5, 7]:
    if k <= len(X_train):
        evaluate(f"k={k}, euclidean, unscaled",
                 KNeighborsClassifier(n_neighbors=k, metric="euclidean"),
                 X_train_unscaled, X_test_unscaled)

results_df = pd.DataFrame(results).drop_duplicates(subset=["Setting"], keep="last")

# KNN Score vs K
plt.figure(figsize=(9, 5))
scaled_results = results_df[
    results_df["Setting"].str.match(r"k=\d+, euclidean, scaled")
].copy()
if not scaled_results.empty:
    scaled_results["k"] = scaled_results["Setting"].str.extract(r"k=(\d+)").astype(int)
    for metric in ["Accuracy", "F1"]:
        plt.plot(scaled_results["k"], scaled_results[metric], marker="o", label=metric)
    plt.xlabel("Number of Neighbours (k)")
    plt.ylabel("Score")
    plt.title("KNN Score vs k (Euclidean, Scaled)")
    plt.legend()
    plt.tight_layout()
    plt.show()

## 11. Final Model Selection

In [ ]:
results_df = results_df.sort_values("F1", ascending=False).reset_index(drop=True)

print("ALL MODEL RESULTS:")
display(results_df)

plt.figure(figsize=(12, 6))
results_df.set_index("Setting")[["Accuracy", "Precision", "Recall", "F1"]].plot(
    kind="bar", ax=plt.gca()
)
plt.title("Comparison of KNN Settings")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.xticks(rotation=75, ha="right")
plt.tight_layout()
plt.show()

best_setting  = results_df.iloc[0]["Setting"]
print("\nBest Setting by Macro F1:", best_setting)

best_unscaled = "unscaled" in best_setting
best_k        = int(best_setting.split(",")[0].split("=")[1])
best_metric   = "manhattan" if "manhattan" in best_setting else "euclidean"

final_model   = KNeighborsClassifier(n_neighbors=best_k, metric=best_metric)
final_X_train = X_train_unscaled if best_unscaled else X_train_scaled
final_X_test  = X_test_unscaled  if best_unscaled else X_test_scaled

final_model.fit(final_X_train, y_train)
final_pred = final_model.predict(final_X_test)
final_cm   = confusion_matrix(y_test, final_pred, labels=np.arange(len(le_y.classes_)))

ConfusionMatrixDisplay(
    confusion_matrix=final_cm, display_labels=le_y.classes_
).plot(cmap="Blues", values_format="d")
plt.title("Final KNN Confusion Matrix")
plt.tight_layout()
plt.show()

print("\nFINAL CLASSIFICATION REPORT:")
print(classification_report(
    y_test, final_pred,
    labels=np.arange(len(le_y.classes_)),
    target_names=[str(c) for c in le_y.classes_],
    zero_division=0
))
print("Final Accuracy:", round(accuracy_score(y_test, final_pred), 4))
print("Final Macro F1:", round(f1_score(y_test, final_pred, average="macro", zero_division=0), 4))

## 12. Predict Sample Inputs

In [ ]:
new_inputs = X.iloc[:3].copy()

for col in new_inputs.columns:
    if new_inputs[col].nunique() > 1:
        spread = X[col].std()
        if pd.notna(spread) and spread > 0:
            new_inputs.iloc[1, new_inputs.columns.get_loc(col)] += 0.10 * spread
            new_inputs.iloc[2, new_inputs.columns.get_loc(col)] -= 0.10 * spread

new_inputs   = new_inputs.reindex(columns=X.columns, fill_value=0).astype(float)
new_selected = selector.transform(new_inputs)
new_scaled   = scaler.transform(new_selected)

prediction_input  = new_selected if best_unscaled else new_scaled
predicted_encoded = final_model.predict(prediction_input)
predicted_labels  = le_y.inverse_transform(predicted_encoded)

print("THREE SAMPLE INPUTS:")
display(new_inputs)

print("\nPREDICTED EXPERIENCE LEVELS:")
for i, label in enumerate(predicted_labels, start=1):
    print(f"Example {i}: Experience Level = {label}")

## 13. Save Results

In [ ]:
results_df.to_csv("knn_model_comparison.csv", index=False)
print("Model comparison saved to: knn_model_comparison.csv")
print("\n" + "=" * 60)
print("ALL STEPS COMPLETED SUCCESSFULLY!")
print("=" * 60)